In [1]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file from the current directory
LOCAL: bool = False

In [2]:
from pyspark.sql import SparkSession

try:
  from databricks.connect import DatabricksSession
  spark = DatabricksSession.builder.getOrCreate()
except Exception as e:
  print("Error creating DatabricksSession:", e)
  spark = SparkSession.builder.getOrCreate()
  LOCAL = True
  print("Fallback to SparkSession successful")

Error creating DatabricksSession: No module named 'databricks'


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 09:28:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Fallback to SparkSession successful


## constants

In [3]:
schema = "`safety-pvprototypes_bronze_dev`.safety_surveillance"

if LOCAL:
  path_narratives = f"data/faers/faers_narratives_small"
else:
  path_narratives = f"/Volumes/safety-pvprototypes_bronze_dev/safety_surveillance/filesets/faers/narratives/faers_narratives"


SEQUENCE_SEPARATOR = "<|endoftext|>" # [EOS]

DEFAULT_VOCABULARY_SIZE = 50257
DEFAULT_EMBEDDINGS_DIMENSION = 256
DEFAULT_SEQUENCE_LENGTH = 4
DEFAULT_STRIDE = 1

## functions

In [4]:
from datetime import datetime, timezone
from pyspark.sql import functions as F, DataFrame
from delta.tables import DeltaTable
from pyspark.errors import AnalysisException
import torch

torch.manual_seed(123)

In [5]:
from pyspark.sql import functions as F, Window


def create_drug_sections(drug_table: str) -> DataFrame:

    df_drug = spark.table(drug_table)
  
    # --- role_cod human-readable mapping ---
    role_map = {
        "PS": "primary suspect",
        "SS": "secondary suspect",
        "C": "concomitant",
        "I": "interacting",
    }

    # --- 1. aggregate drugs per case ---
    df_drugs_agg = (
        df_drug
        .withColumn(
            "drug_desc",
            F.concat_ws(
                " ",
                F.upper(F.col("drugname")),
                F.when(F.col("dose_vbm").isNotNull(), F.concat(F.lit("("), F.col("dose_vbm"), F.lit(")"))),
                F.when(F.col("route").isNotNull(), F.concat(F.lit("via "), F.lower(F.col("route")))),
            ),
        )
        .withColumn(
            "role_desc",
            F.coalesce(
                F.create_map(*[item for kv in role_map.items() for item in (F.lit(kv[0]), F.lit(kv[1]))])[F.col("role_cod")],
                F.col("role_cod"),
            ),
        )
        .withColumn("drug_text", F.concat(F.col("drug_desc"), F.lit(" ["), F.col("role_desc"), F.lit("]")))
        .groupBy("primaryid")
        .agg(F.concat_ws("; ", F.collect_list("drug_text")).alias("drugs_text"))
    )

    return df_drugs_agg


def create_reaction_sections(reactions_table: str) -> DataFrame:

    df_reactions = spark.table(reactions_table)

    # --- 2. aggregate reactions per case ---
    df_reac_agg = (
        df_reactions
        .withColumn(
            "reac_text",
            F.concat_ws(
                " ",
                F.lower(F.col("pt")),
                F.when(F.col("pt_code").isNotNull(), F.concat(F.lit("(PT:"), F.col("pt_code"), F.lit(")"))),
                F.when(F.col("pt_soc_code").isNotNull(), F.concat(F.lit("[SOC:"), F.col("pt_soc_code"), F.lit("]"))),
                F.when(
                    F.col("nnmq_codes").isNotNull() & (F.size("nnmq_codes") > 0),
                    F.concat(F.lit("NNMQ:"), F.concat_ws(",", F.col("nnmq_codes"))),
                ),
                F.when(
                    F.col("smq_codes").isNotNull() & (F.size("smq_codes") > 0),
                    F.concat(F.lit("SMQ:"), F.concat_ws(",", F.col("smq_codes"))),
                ),
            ),
        )
        .groupBy("primaryid")
        .agg(F.concat_ws("; ", F.collect_list("reac_text")).alias("reactions_text"))
    )
    return df_reac_agg


def create_demo_sections(demo_table: str) -> DataFrame:

    df_demo = spark.table(demo_table)
    # --- 3. build demo text fields ---
    df_demo_text = (
        df_demo
        .withColumn(
            "age_text",
            F.when(
                F.col("age").isNotNull(),
                F.concat(
                    F.col("age").cast("string"),
                    F.lit("-"),
                    F.coalesce(
                        F.create_map(
                            F.lit("YR"), F.lit("year-old"),
                            F.lit("MON"), F.lit("month-old"),
                            F.lit("WK"), F.lit("week-old"),
                            F.lit("DY"), F.lit("day-old"),
                            F.lit("HR"), F.lit("hour-old"),
                            F.lit("DEC"), F.lit("decade-old"),
                        )[F.col("age_cod")],
                        F.lit("year-old"),
                    ),
                ),
            ).otherwise(F.lit("age unknown")),
        )
        .withColumn(
            "sex_text",
            F.coalesce(
                F.create_map(
                    F.lit("F"), F.lit("female"),
                    F.lit("M"), F.lit("male"),
                    F.lit("UNK"), F.lit("sex unknown"),
                    F.lit("NS"), F.lit("sex not specified"),
                )[F.col("sex")],
                F.lit("sex unknown"),
            ),
        )
        .withColumn(
            "rept_dt_text",
            F.when(
                F.col("rept_dt").isNotNull() & (F.length(F.col("rept_dt").cast("string")) == 8),
                F.date_format(F.to_date(F.col("rept_dt").cast("string"), "yyyyMMdd"), "yyyy-MM-dd"),
            )
            .when(F.col("rept_dt").isNotNull() & (F.length(F.col("rept_dt").cast("string")) == 6), 
                F.date_format(F.to_date(F.concat(F.col("rept_dt").cast("string"), F.lit("01")).cast("string"), "yyyyMMdd"), "yyyy-MM-dd")
            )
            .otherwise(F.lit("date unknown")),
        )
        .select("primaryid", "age_text", "sex_text", "rept_dt_text", "period")
    )

    return df_demo_text


def create_narratives(demo_table: str, drug_table: str, reactions_table: str) -> DataFrame:

    df_demo_text = create_demo_sections(demo_table)
    df_drugs_agg = create_drug_sections(drug_table)
    df_reac_agg = create_reaction_sections(reactions_table)

    # --- 4. join and format as narrative ---
    df_narratives = (
        df_demo_text
        .join(df_drugs_agg, on="primaryid", how="left")
        .join(df_reac_agg, on="primaryid", how="left")
        .withColumn(
            "text",
            F.concat_ws(
                " ",
                F.lit("FAERS case"),
                F.col("primaryid").cast("string"),
                F.concat(F.lit("(period "), F.col("period"), F.lit("):")),
                F.col("age_text"),
                F.concat(F.col("sex_text"), F.lit(",")),
                F.concat(F.lit("reported "), F.col("rept_dt_text"), F.lit(".")),
                F.concat(F.lit("Medications: "), F.coalesce(F.col("drugs_text"), F.lit("none reported")), F.lit(".")),
                F.concat(F.lit("Adverse reactions: "), F.coalesce(F.col("reactions_text"), F.lit("none reported")), F.lit(".")),
            ),
        )
        .select("primaryid", "text")
    )

    return df_narratives

def get_corpus_subset(subset_length: int=10000) -> str:
    
    if LOCAL:
        df_narratives = spark.read.load(path_narratives)
    else:
        df_narratives = spark.read.json(path_narratives)
    result: str = SEQUENCE_SEPARATOR.join(df_narratives.limit(subset_length).select("text").toPandas()["text"].tolist())
    
    return result
  

### data corpus

In [6]:
# df_narratives = create_narratives(demo_table=f"{schema}.silver_demo", drug_table=f"{schema}.silver_drug", reactions_table=f"{schema}.silver_reac")
# df_narratives.coalesce(1).write.mode("overwrite").json(path_narratives)

### data preparation stage

In [7]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer: tiktoken.Encoding|None = None, max_length: int = DEFAULT_SEQUENCE_LENGTH, stride: int = DEFAULT_STRIDE):
    self.input_ids = []
    self.target_ids = []
    if tokenizer is None:
      tokenizer = tiktoken.get_encoding("gpt2")
    token_ids = tokenizer.encode(txt, allowed_special={SEQUENCE_SEPARATOR})
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i + max_length]
      target_chunk = token_ids[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)
  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]
  
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
  dataset = GPTDatasetV1(txt, max_length=max_length, stride=stride)
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
  return dataloader

class BatchToEmbeddings:

  def __init__(self,
                output_dimension=DEFAULT_EMBEDDINGS_DIMENSION,
                vocabulary_size=DEFAULT_VOCABULARY_SIZE,
                sequence_length=DEFAULT_SEQUENCE_LENGTH):

    self._text_embeddings = torch.nn.Embedding(vocabulary_size, output_dimension)
    position_embedding = torch.nn.Embedding(sequence_length, output_dimension)
    self._position_embeddings = position_embedding(torch.arange(sequence_length))

  def embed(self, inputs, targets):

    input_embeddings = self._text_embeddings(inputs) + self._position_embeddings
    target_embeddings = self._text_embeddings(targets) + self._position_embeddings

    return input_embeddings, target_embeddings

class GPTEmbeddedDatasetV1(Dataset):
  def __init__(self, txt, tokenizer: tiktoken.Encoding|None = None, 
              sequence_length: int = DEFAULT_SEQUENCE_LENGTH, 
              stride: int = DEFAULT_STRIDE,
              output_dimension=DEFAULT_EMBEDDINGS_DIMENSION,
              vocabulary_size=DEFAULT_VOCABULARY_SIZE,
               ):
    
    self.input_ids = []
    self.target_ids = []
    if tokenizer is None:
      tokenizer = tiktoken.get_encoding("gpt2")
    
    self._text_embeddings = torch.nn.Embedding(vocabulary_size, output_dimension)
    position_embedding = torch.nn.Embedding(sequence_length, output_dimension)
    self._position_embeddings = position_embedding(torch.arange(sequence_length))

    token_ids = tokenizer.encode(txt, allowed_special={SEQUENCE_SEPARATOR})
    for i in range(0, len(token_ids) - sequence_length, stride):
      input_chunk = token_ids[i:i + sequence_length]
      target_chunk = token_ids[i + 1: i + sequence_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)
  def __getitem__(self, idx):
    _x = self.input_ids[idx]
    _y = self.target_ids[idx]

    input_embeddings = self._text_embeddings(_x) + self._position_embeddings
    target_embeddings = self._text_embeddings(_y) + self._position_embeddings

    return input_embeddings, target_embeddings
  
def create_dataloader_v2(txt, batch_size=4, sequence_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
  dataset = GPTEmbeddedDatasetV1(txt, sequence_length=sequence_length, stride=stride)
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
  return dataloader

In [8]:
corpus = get_corpus_subset()
dataloader = create_dataloader_v1(corpus[:1000], batch_size=8, max_length=4, stride=1, shuffle=False)
embeddings = BatchToEmbeddings()
dataloader_iterator = iter(dataloader)

for i in range(2):
  input_ids, target_ids = next(dataloader_iterator)
  print(f"*** Batch {i+1} - Input IDs: {input_ids} | Target IDs: {target_ids}")
  input, target = embeddings.embed(input_ids, target_ids)
  print(f"***Input embeddings shape: {input.shape} | Target embeddings shape: {target.shape}")
  print(f"*** Batch {i+1} - Input IDs: {input} | Target IDs: {target}")

*** Batch 1 - Input IDs: tensor([[ 7708,  4877,  1339, 12279],
        [ 4877,  1339, 12279, 42548],
        [ 1339, 12279, 42548, 38056],
        [12279, 42548, 38056,   357],
        [42548, 38056,   357, 41007],
        [38056,   357, 41007,  1315],
        [  357, 41007,  1315,    80],
        [41007,  1315,    80,    18]]) | Target IDs: tensor([[ 4877,  1339, 12279, 42548],
        [ 1339, 12279, 42548, 38056],
        [12279, 42548, 38056,   357],
        [42548, 38056,   357, 41007],
        [38056,   357, 41007,  1315],
        [  357, 41007,  1315,    80],
        [41007,  1315,    80,    18],
        [ 1315,    80,    18,  2599]])
***Input embeddings shape: torch.Size([8, 4, 256]) | Target embeddings shape: torch.Size([8, 4, 256])
*** Batch 1 - Input IDs: tensor([[[ 0.6118,  0.8430,  0.6591,  ...,  1.4890, -1.8595,  1.2459],
         [ 0.4971, -0.0859, -1.8633,  ..., -0.0118,  1.1630, -1.7880],
         [-0.3202,  0.8756,  0.9386,  ...,  0.3237, -1.2675, -1.6838],
         [ 

In [9]:
torch.manual_seed(123)

corpus = get_corpus_subset()
dataloader = create_dataloader_v2(corpus[:1000], batch_size=4, sequence_length=4, stride=1, shuffle=False)
dataloader_iterator = iter(dataloader)

# for i in range(2):
#   input, target = next(dataloader_iterator)
#   print(f"***Input embeddings shape: {input.shape} | Target embeddings shape: {target.shape}")
#   print(f"*** Batch {i+1} - Input IDs: {input} | Target IDs: {target}")

### attention mechanism

In [10]:
corpus = get_corpus_subset()
dataloader = create_dataloader_v2(corpus[:1000], batch_size=4, sequence_length=4, stride=1, shuffle=False)
dataloader_iterator = iter(dataloader)

input_batch, target_batch = next(dataloader_iterator)
print(f"***Input embeddings shape: {input_batch.shape} | Target embeddings shape: {target_batch.shape}")
print(f"*** Batch 1 - Input IDs: {input_batch} | Target IDs: {target_batch}")

***Input embeddings shape: torch.Size([4, 4, 256]) | Target embeddings shape: torch.Size([4, 4, 256])
*** Batch 1 - Input IDs: tensor([[[ 1.6732, -1.3500, -0.4106,  ...,  0.2176,  2.2215,  1.7704],
         [-0.7278,  0.8332,  1.0601,  ..., -2.0067, -0.2726, -0.8512],
         [-1.0492, -0.2573,  1.2796,  ...,  1.1272,  0.2612,  1.6815],
         [-0.7112,  1.8577,  1.1139,  ...,  2.6052, -2.2392, -0.0793]],

        [[ 0.2000,  1.0974, -0.0677,  ..., -1.1911,  2.5624,  0.3596],
         [ 0.2407,  2.0377,  0.3055,  ...,  0.9077, -0.2302, -0.4400],
         [-1.2452, -0.9595,  1.8721,  ...,  1.3580, -0.3056,  1.0703],
         [-1.3370, -0.4747,  2.0964,  ...,  0.2389, -3.8522, -0.0352]],

        [[ 1.1685,  2.3020, -0.8223,  ...,  1.7232,  2.6048,  0.7709],
         [ 0.0447,  1.3356,  0.8980,  ...,  1.1385, -0.7969, -1.0511],
         [-1.8710, -3.2919,  2.8547,  ..., -1.0084, -1.9186,  1.1145],
         [-0.6384,  0.9619, -0.0784,  ...,  0.3559, -2.9217,  1.1117]],

        [[ 0.97

#### computing attention weights

In [11]:
def compute_attention_weights(batch) -> torch.Tensor:
  
  n_sequences, sequence_length, embedding_dimension = batch.shape

  attention_scores = torch.empty(n_sequences, sequence_length, sequence_length)
  for idx, input in enumerate(batch):
    for i, xi in enumerate(input):
      for j, xj in enumerate(input):
        attention_scores[idx, i, j] = torch.dot(xi, xj)

  return torch.softmax(attention_scores, dim=-1)


def compute_context_vectors(batch, attention_weights) -> torch.Tensor:

  context_vector = torch.zeros(batch.shape[0], batch.shape[2])
  for idx, input in enumerate(batch):
    for i, xi in enumerate(input):
      context_vector += attention_weights[idx, i].unsqueeze(1) * xi
  
  return context_vector


In [12]:
attention_weights = compute_attention_weights(input_batch)
print(f"Attention weights shape: {attention_weights.shape}")
print(f"Attention weights: {attention_weights}")
context_vectors = compute_context_vectors(input_batch, attention_weights)
print(f"Context vectors shape: {context_vectors.shape}")
print(f"Context vectors: {context_vectors}")


Attention weights shape: torch.Size([4, 4, 4])
Attention weights: tensor([[[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.]],

        [[1., 0., 0., 0.],
         [0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 0., 1.]]], grad_fn=<SoftmaxBackward0>)
Context vectors shape: torch.Size([4, 256])
Context vectors: tensor([[  4.0142,   3.6492,  -1.5305,  ...,   2.7036,   9.4268,   3.0605],
        [ -1.0233,   3.2097,   4.1442,  ...,  -1.1883,  -3.7096,  -3.3493],
        [ -5.3378,  -6.3640,   6.6863,  ...,   0.5854,  -2.9511,   6.1277],
        [ -2.3086,   3.3813,   4.7946,  ...,   4.1571, -12.4082,   0.6456]],
       grad_fn=<AddBackward0>)


#### self-attention with trainable weights

In [13]:
corpus = get_corpus_subset()
dataloader = create_dataloader_v2(corpus[:1000], batch_size=4, sequence_length=4, stride=1, shuffle=False)
dataloader_iterator = iter(dataloader)

In [14]:
input_batch, target_batch = next(dataloader_iterator)
# print(input_batch)
print(input_batch.shape)
# print(target_batch)
print(target_batch.shape)

qkv_dimension = 2
n_sequences, sequence_length, embedding_dimension = input_batch.shape

W_query = torch.nn.Parameter(torch.rand(embedding_dimension, qkv_dimension), requires_grad=False) # 256/2
W_key = torch.nn.Parameter(torch.rand(embedding_dimension, qkv_dimension), requires_grad=False) # 256/2
W_value = torch.nn.Parameter(torch.rand(embedding_dimension, qkv_dimension), requires_grad=False) # 256/2

sequence_context_vector = torch.empty(n_sequences, sequence_length, qkv_dimension) # a context vector for each sequence 

for idx, sequence in enumerate(input_batch):
  query_context_vector = torch.empty(sequence_length, qkv_dimension) # attention weights for each token in the sequence attending to each token in the sequence
  for i, toquen_query in enumerate(sequence):

    query = toquen_query @ W_query # 1/256 @ 256/2 -> 1/2
    keys = torch.empty(sequence_length, qkv_dimension) # a key for each query of each token in each sequence
    values = torch.empty(sequence_length, qkv_dimension) # a value for each query of each token in each sequence
    for j, token in enumerate(sequence):
      keys[j] = token @ W_key # 1/256 @ 256/2 -> 1/2
      values[j] = token @ W_value # 1/256 @ 256/2 -> 1/2

    query_attention_scores = query @ keys.T
    query_attention_weights = torch.softmax(query_attention_scores / (qkv_dimension ** 0.5), dim=-1)
    query_context_vector[i] = query_attention_weights @ values
    
  sequence_context_vector[idx] = query_context_vector

print(f"Sequence context vectors shape: {sequence_context_vector.shape}")
print(f"Sequence context vectors: {sequence_context_vector}") 

torch.Size([4, 4, 256])
torch.Size([4, 4, 256])
Sequence context vectors shape: torch.Size([4, 4, 2])
Sequence context vectors: tensor([[[ -0.6857,  12.9033],
         [ -0.4251,  12.8384],
         [ -0.6857,  12.9033],
         [  6.2114,  -0.0950]],

        [[  2.7551,  -2.0886],
         [ 35.1768,  10.6169],
         [ 35.1768,  10.6169],
         [ -8.1429, -22.2915]],

        [[ 17.2859,   5.8622],
         [ 27.3187,  15.0076],
         [-11.5876, -24.2635],
         [ 27.3187,  15.0076]],

        [[-13.5985,   3.1126],
         [ 24.1295,  12.8476],
         [ 28.1968,   9.8565],
         [ 28.1834,   9.8664]]], grad_fn=<CopySlices>)


In [15]:
#input_batch, target_batch = next(dataloader_iterator)
# print(input_batch)
print(input_batch.shape)
# print(target_batch)
print(target_batch.shape)

qkv_dimension = 2
n_sequences, sequence_length, embedding_dimension = input_batch.shape

#W_query = torch.nn.Parameter(torch.rand(embedding_dimension, qkv_dimension), requires_grad=False) # 256/2
#W_key = torch.nn.Parameter(torch.rand(embedding_dimension, qkv_dimension), requires_grad=False) # 256/2
#W_value = torch.nn.Parameter(torch.rand(embedding_dimension, qkv_dimension), requires_grad=False) # 256/2

sequence_context_vector = torch.empty(n_sequences, sequence_length, qkv_dimension) # a context vector for each sequence 

for idx, sequence in enumerate(input_batch):
  query_context_vector = torch.empty(sequence_length, qkv_dimension) # attention weights for each token in the sequence attending to each token in the sequence
  
  keys = sequence @ W_key # 4/256 @ 256/2 -> 4/2
  values = sequence @ W_value # 4/256 @ 256/2 -> 4/2
  queries = sequence @ W_query # 4/256 @ 256/2 -> 4/2
  query_attention_scores = queries @ keys.T
  query_attention_weights = torch.softmax(query_attention_scores / (qkv_dimension ** 0.5), dim=-1)
  query_context_vector = query_attention_weights @ values
  sequence_context_vector[idx] = query_context_vector

print(f"Sequence context vectors shape: {sequence_context_vector.shape}")
print(f"Sequence context vectors: {sequence_context_vector}") 

torch.Size([4, 4, 256])
torch.Size([4, 4, 256])
Sequence context vectors shape: torch.Size([4, 4, 2])
Sequence context vectors: tensor([[[ -0.6857,  12.9033],
         [ -0.4251,  12.8384],
         [ -0.6857,  12.9033],
         [  6.2114,  -0.0950]],

        [[  2.7551,  -2.0886],
         [ 35.1768,  10.6169],
         [ 35.1768,  10.6169],
         [ -8.1429, -22.2915]],

        [[ 17.2859,   5.8622],
         [ 27.3187,  15.0076],
         [-11.5876, -24.2635],
         [ 27.3187,  15.0076]],

        [[-13.5985,   3.1126],
         [ 24.1295,  12.8476],
         [ 28.1968,   9.8565],
         [ 28.1834,   9.8664]]], grad_fn=<CopySlices>)


#### compact self-attention class

In [16]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):
  def __init__(self, embedding_dimension, qkv_dimension, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)
    self.W_key = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)
    self.W_value = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)

  def forward(self, x):

    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    context_vec = attn_weights @ values
    return context_vec

In [17]:

sa = SelfAttention_v2(embedding_dimension=embedding_dimension, qkv_dimension=qkv_dimension)
sequence_context_vector = sa.forward(input_batch[0])


print(f"Sequence context vectors shape: {sequence_context_vector.shape}")
print(f"Sequence context vectors: {sequence_context_vector}") 

Sequence context vectors shape: torch.Size([4, 2])
Sequence context vectors: tensor([[-0.3333, -0.1651],
        [-0.2641,  0.0582],
        [-0.3356, -0.0159],
        [-0.2015,  0.0037]], grad_fn=<MmBackward0>)


#### applying causal attention mask

In [18]:
import torch.nn as nn

class SelfAttention_v3(nn.Module):
  def __init__(self, embedding_dimension, qkv_dimension, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)
    self.W_key = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)
    self.W_value = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)

  def forward(self, x):

    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attention_scores = queries @ keys.T
    context_length = attention_scores.shape[0]
    mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
    masked_attention_scores = attention_scores.masked_fill(mask.bool(), -torch.inf)
    attn_weights = torch.softmax(masked_attention_scores / keys.shape[-1]**0.5, dim=-1)
    context_vec = attn_weights @ values
    return context_vec

In [19]:
sa = SelfAttention_v3(embedding_dimension=embedding_dimension, qkv_dimension=qkv_dimension)
context_vector = sa.forward(input_batch[0])
context_vector

tensor([[-0.4874, -0.3632],
        [ 0.0566, -0.4391],
        [-0.5913,  0.0635],
        [-0.1234,  0.1212]], grad_fn=<MmBackward0>)

#### masking additional attention weights with dropout

In [20]:
import torch.nn as nn

class SelfAttention_v4(nn.Module):
  def __init__(self, embedding_dimension, qkv_dimension, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)
    self.W_key = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)
    self.W_value = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)

  def forward(self, x):

    keys = self.W_key(x) # seq_len/embedding_dimension @ embedding_dimension/qkv_dimension -> seq_len/qkv_dimension
    queries = self.W_query(x)
    values = self.W_value(x)

    attention_scores = queries @ keys.T # seq_len/qkv_dimension @ qkv_dimension/seq_len -> seq_len/seq_len
    context_length = x.shape[0] # seq_len
    mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
    masked_attention_scores = attention_scores.masked_fill(mask.bool(), -torch.inf)
    attn_weights = torch.softmax(masked_attention_scores / keys.shape[-1]**0.5, dim=-1)

    dropout = torch.nn.Dropout(0.5)
    attn_weights_after_dropout = dropout(attn_weights)

    context_vec = attn_weights_after_dropout @ values
    return context_vec

In [21]:
sa = SelfAttention_v4(embedding_dimension=embedding_dimension, qkv_dimension=qkv_dimension)
context_vector = sa.forward(input_batch[0])
context_vector

tensor([[ 0.0000,  0.0000],
        [ 2.2239, -2.2561],
        [ 0.7559, -0.3506],
        [ 0.2746,  0.1248]], grad_fn=<MmBackward0>)

#### compact causal attention class

In [22]:
import torch.nn as nn

class CausalAttention(nn.Module):
  def __init__(self, embedding_dimension, qkv_dimension, context_length: int, qkv_bias=False, dropout: float=0.5):
    super().__init__()
    self.embedding_dimension = embedding_dimension
    self.qkv_dimension = qkv_dimension
    self.context_length = context_length
    self.W_query = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)
    self.W_key = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)
    self.W_value = nn.Linear(embedding_dimension, qkv_dimension, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):

    n_sequences, sequence_length, embedding_dimension = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attention_scores = queries @ keys.transpose(-2, -1)
    masked_attention_scores = attention_scores.masked_fill(self.mask.bool()[:sequence_length, :sequence_length], -torch.inf)
    attn_weights = torch.softmax(masked_attention_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights_after_dropout = self.dropout(attn_weights)

    context_vec = attn_weights_after_dropout @ values
    return context_vec

In [23]:
sa = CausalAttention(embedding_dimension=embedding_dimension, qkv_dimension=qkv_dimension, context_length=sequence_length)
context_vector = sa(input_batch)
context_vector

tensor([[[ 1.0845,  2.0374],
         [ 0.5179,  0.9243],
         [-0.6977,  0.8572],
         [-0.2324,  0.5592]],

        [[-1.0835,  0.1859],
         [ 0.0000,  0.0000],
         [ 0.6523,  0.6084],
         [ 2.0785,  0.1029]],

        [[-0.0966,  1.2138],
         [ 0.0000,  0.0000],
         [ 0.2638,  0.5491],
         [ 0.5221, -0.4151]],

        [[-1.7598,  1.0333],
         [ 2.6797,  1.0984],
         [ 0.5538,  0.7756],
         [ 1.7433, -1.4073]]], grad_fn=<UnsafeViewBackward0>)

In [34]:
print(context_vector.shape)
print(context_vector.transpose(0, 2).shape)

torch.Size([4, 4, 2])
torch.Size([2, 4, 4])


#### multi-head attention

In [26]:
class MultiHeadAttentionWrapper(nn.Module):

  def __init__(self, embedding_dimension, qkv_dimension, context_length: int, heads: int, qkv_bias=False, dropout: float=0.5):
    super().__init__()
    self.heads = nn.ModuleList(
      [
        CausalAttention(embedding_dimension, qkv_dimension, context_length, dropout=dropout, qkv_bias=qkv_bias)
        for _ in range(heads)
      ]
    )
    
  def forward(self, x):
    return torch.cat([head(x) for head in self.heads], dim=-1)

In [27]:
mha = MultiHeadAttentionWrapper(embedding_dimension, qkv_dimension, context_length=sequence_length, heads=2, dropout=0.0, )
context_vecs = mha(input_batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)



tensor([[[-0.1360,  0.2360,  0.7687,  0.6661],
         [ 1.0884,  0.0785,  0.7140,  0.5950],
         [ 0.1361,  0.0169,  0.6925,  0.7151],
         [-0.1069, -0.0963,  0.6923,  0.4321]],

        [[ 0.3865,  0.1678,  1.1613,  1.0566],
         [ 0.9611,  0.3044,  0.9956,  1.1095],
         [ 0.7547,  0.3413,  1.0195,  0.7408],
         [ 0.5395,  0.4420,  0.8627,  0.4560]],

        [[ 0.1905,  0.4270,  1.0153,  1.7850],
         [ 1.2086,  0.8492,  1.1780,  0.7611],
         [ 0.5115,  0.6059,  1.0936,  0.2124],
         [ 0.7228,  0.7333,  0.4498,  0.0514]],

        [[ 0.6665,  1.0783,  1.7971,  0.5046],
         [ 1.1017,  0.9678,  1.1720, -0.1842],
         [ 0.5255,  0.9858,  1.1188,  0.0026],
         [ 0.5582,  0.8701,  1.2101,  0.1721]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([4, 4, 4])


In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, embedding_dimension, output_dimension, context_length: int, heads: int, qkv_bias=False, dropout: float=0.5):
    super().__init__()
    assert (output_dimension % heads == 0), "output_dimension must be divisible by heads"

    self.embedding_dimension = embedding_dimension
    self.output_dimension = output_dimension
    self.context_length = context_length
    self.head_dimension = output_dimension // heads
    self.heads = heads

    self.W_query = nn.Linear(embedding_dimension, output_dimension, bias=qkv_bias)
    self.W_key = nn.Linear(embedding_dimension, output_dimension, bias=qkv_bias)
    self.W_value = nn.Linear(embedding_dimension, output_dimension, bias=qkv_bias)

    self.out_projection = nn.Linear(output_dimension, output_dimension)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):

    n_sequences, sequence_length, embedding_dimension = x.shape
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    keys = keys.view(n_sequences, sequence_length, self.heads, self.head_dimension)
    values = values.view(n_sequences, sequence_length, self.heads, self.head_dimension)
    queries = queries.view(n_sequences, sequence_length, self.heads, self.head_dimension)

    keys = keys.transpose(1, 2)
    queries = queries.transpose(1, 2)
    values = values.transpose(1, 2)
    # Transposes from shape (n_sequences, sequence_length, heads, head_dimension) to (n_sequences, heads, sequence_length, head_dimension)

    attention_scores = queries @ keys.transpose(-2, -1)
    masked_attention_scores = attention_scores.masked_fill(self.mask.bool()[:sequence_length, :sequence_length], -torch.inf)
    attn_weights = torch.softmax(masked_attention_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights_after_dropout = self.dropout(attn_weights)

    context_vec = self.out_projection((attn_weights_after_dropout @ values).transpose(1, 2).contiguous().view(n_sequences, sequence_length, self.output_dimension))
    return context_vec

In [43]:
mha = MultiHeadAttention(embedding_dimension=embedding_dimension, output_dimension=qkv_dimension, context_length=sequence_length, heads=2, dropout=0.0, )
context_vecs = mha(input_batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 0.3860, -0.0236],
         [ 0.3434,  0.0328],
         [ 0.3969,  0.0616],
         [ 0.2330,  0.0242]],

        [[ 0.6331,  0.1442],
         [ 0.4939,  0.0890],
         [ 0.3240,  0.0666],
         [ 0.1464, -0.0643]],

        [[ 0.5638,  0.0716],
         [ 0.3071, -0.0080],
         [ 0.0832, -0.0343],
         [ 0.2007, -0.0391]],

        [[ 0.2322, -0.0122],
         [-0.0118, -0.1795],
         [ 0.1748, -0.0488],
         [ 0.1930, -0.0379]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([4, 4, 2])


### a GPT model for text generation / LLM architecture

In [48]:
class DummyTransformerBlock(nn.Module):

  def __init__(self, cfg):
    super().__init__()

  def forward(self, x):
    return x

In [55]:
class DummyLayerNorm(nn.Module):

  def __init__(self, normalized_shape, eps=1e-5):
    super().__init__()
  
  def forward(self, x):
    return x
  

class LayerNorm(nn.Module):

  def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + self.eps)
    return self.scale * norm_x + self.shift

In [51]:
class DummyGPTModel(nn.Module):

  def __init__(self, cfg):
    super().__init__()
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb = nn.Dropout(cfg["drop_rate"])
    self.trf_blocks = nn.Sequential(*[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
    self.final_norm = DummyLayerNorm(cfg["emb_dim"])
    self.out_head = nn.Linear(
      cfg["emb_dim"], cfg["vocab_size"], bias=False
    )

  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(
      torch.arange(seq_len, device=in_idx.device)
    )
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

#### create dummy text batch

In [54]:
GPT_CONFIG_124M = {
  "vocab_size": 50257, # Vocabulary size
  "context_length": 1024, # Context length
  "emb_dim": 768, # Embedding dimension
  "n_heads": 12, # Number of attention heads
  "n_layers": 12, # Number of layers (transformer blocks)
  "drop_rate": 0.1, # Dropout rate
  "qkv_bias": False # Query-Key-Value bias
}

In [52]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


#### create dummy model

In [53]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


##### adding the normalization layer

In [59]:
batch_example = torch.randn(2, 5)

ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[-2.9802e-08],
        [-1.1921e-08]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


#### Implementing a feed forward network with GELU activations

In [ ]:
class GELU(nn.Module):

  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) *(x + 0.044715 * torch.pow(x, 3))))

In [61]:
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
      GELU(),
      nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
    )
  def forward(self, x):
    return self.layers(x)

In [62]:
ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2, 3, 768)
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])


#### adding shortcut connections

In [63]:
class ExampleDeepNeuralNetwork(nn.Module):

  def __init__(self, layer_sizes, use_shortcut):
    super().__init__()
    self.use_shortcut = use_shortcut
    self.layers = nn.ModuleList([
      nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]),
      GELU()),
      nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]),
      GELU()),
      nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]),
      GELU()),
      nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]),
      GELU()),
      nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]),
      GELU())
    ])

  def forward(self, x):
    for layer in self.layers:
      layer_output = layer(x)
      if self.use_shortcut and x.shape == layer_output.shape:
        x = x + layer_output
      else:
        x = layer_output
    return x

In [68]:
layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=False)

In [69]:
def print_gradients(model, x):
  output = model(x)
  target = torch.tensor([[0.]])
  loss = nn.MSELoss()
  loss = loss(output, target)
  loss.backward()
  for name, param in model.named_parameters():
    if 'weight' in name:
      print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [70]:
print_gradients(model_without_shortcut, sample_input)
# the gradients become smaller
# as we progress from the last layer (layers.4) to the first layer (layers.0), which is
# a phenomenon called the vanishing gradient problem.


layers.0.0.weight has gradient mean of 0.00020173584925942123
layers.1.0.weight has gradient mean of 0.00012011158833047375
layers.2.0.weight has gradient mean of 0.0007152041071094573
layers.3.0.weight has gradient mean of 0.0013988735154271126
layers.4.0.weight has gradient mean of 0.005049645435065031


In [71]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=True)
print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.22169791162014008
layers.1.0.weight has gradient mean of 0.20694106817245483
layers.2.0.weight has gradient mean of 0.32896992564201355
layers.3.0.weight has gradient mean of 0.2665732204914093
layers.4.0.weight has gradient mean of 1.3258540630340576


#### Connecting attention and linear layers in a transformer block

In [ ]:
# ...